# Semana 11: Salva al cliente: clasificación

**Introducción a la Ciencia de Datos** · Dudbil Olvasada Pabón Riaño · UNAB

Cuaderno guiado. Cada celda corresponde a un ejercicio de la guía de laboratorio; los resultados deben coincidir con sus puntos de control. Los datos se leen desde el repositorio del curso.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
BASE = "https://raw.githubusercontent.com/dudbilopr/Introduccion-a-la-ciencia-de-datos/main/Semanas/Datos/"

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
df = pd.read_csv(BASE + "churn_telecom.csv")
df.columns = [" ".join(c.split()) for c in df.columns]
y, X = df["Churn"], df.drop(columns="Churn")

## 1. Explorar las señales

In [ ]:
print("Clientes", len(df), " se van", y.sum(), " tasa", round(y.mean(), 4))
for col in ["Complains", "Status", "Tariff Plan"]:
    print(df.groupby(col)["Churn"].agg(["count", "mean"]).round(3))
print("Fallas (media):", df.groupby("Churn")["Call Failure"].mean().round(1).to_dict())
print("Segundos (mediana):", df.groupby("Churn")["Seconds of Use"].median().to_dict())

## 2. La línea base

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print("Prueba:", len(yte), " se van:", yte.sum(), " exactitud de 'nadie se va':", round(1 - yte.mean(), 3))

## 3. Regresión logística

In [ ]:
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(Xtr, ytr)
prob = lr.predict_proba(Xte)[:, 1]
for u in [0.5, 0.3, 0.2]:
    p = (prob >= u).astype(int)
    print(u, "exactitud", round(accuracy_score(yte, p), 3), " precisión", round(precision_score(yte, p), 3),
          " sensibilidad", round(recall_score(yte, p), 3), " matriz", confusion_matrix(yte, p).tolist())

## 4. Árbol de decisión

In [ ]:
arbol = DecisionTreeClassifier(max_depth=3, random_state=42).fit(Xtr, ytr)
pa = arbol.predict(Xte)
print(export_text(arbol, feature_names=list(X.columns), decimals=1))
print("exactitud", round(accuracy_score(yte, pa), 3), " precisión", round(precision_score(yte, pa), 3),
      " sensibilidad", round(recall_score(yte, pa), 3), " matriz", confusion_matrix(yte, pa).tolist())

## 5. Reto: 150 llamadas
Cada cliente salvado vale 10 puntos; cada llamada cuesta 1 (supuestos del juego).

In [ ]:
orden = np.argsort(-prob)
encontrados = yte.values[orden[:150]].sum()
print("Top 150: encontrados", encontrados, " puntos", encontrados * 10 - 150, " corte de probabilidad", round(prob[orden[149]], 3))
p5 = (prob >= 0.5).astype(int)
print("Umbral 0,5: llamadas", p5.sum(), " puntos", (p5 & yte.values).sum() * 10 - p5.sum())

## Declaración de uso de IA

Si usó una herramienta de IA, escriba aquí: cuál, para qué, el prompt, la versión del modelo y qué verificó usted.